# Model Saving

This notebook trains the final Random Forest model and saves the complete machine learning pipeline for use in the Student Performance Prediction and Academic Advisory System.

The saved pipeline includes:

- Data preprocessing
- Numerical feature scaling
- Categorical feature encoding
- Random Forest classification model

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [2]:
# Load the student mathematics dataset
df = pd.read_csv("../data/raw/student-mat.csv", sep=";")

# Display the first five rows
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [3]:
def categorize_performance(grade):
    if grade <= 9:
        return "At Risk"
    elif grade <= 14:
        return "Average"
    else:
        return "High Performance"


df["performance_category"] = df["G3"].apply(categorize_performance)

df[["G3", "performance_category"]].head(10)

,G3,performance_category
0,6,At Risk
1,6,At Risk
2,10,Average
3,15,High Performance
4,10,Average
5,15,High Performance
6,11,Average
7,6,At Risk
8,19,High Performance
9,15,High Performance


In [4]:
# Define input features
X = df.drop(
    columns=[
        "G1",
        "G2",
        "G3",
        "performance_category"
    ]
)

# Define target
y = df["performance_category"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (395, 30)
Target shape: (395,)


In [5]:
categorical_columns = [
    "school",
    "sex",
    "address",
    "famsize",
    "Pstatus",
    "Mjob",
    "Fjob",
    "reason",
    "guardian",
    "schoolsup",
    "famsup",
    "paid",
    "activities",
    "nursery",
    "higher",
    "internet",
    "romantic"
]

numerical_columns = [
    "age",
    "Medu",
    "Fedu",
    "traveltime",
    "studytime",
    "failures",
    "famrel",
    "freetime",
    "goout",
    "Dalc",
    "Walc",
    "health",
    "absences"
]

print("Number of categorical features:", len(categorical_columns))
print("Number of numerical features:", len(numerical_columns))

Number of categorical features: 17
Number of numerical features: 13


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training features: (316, 30)
Testing features: (79, 30)


In [7]:
# Preprocessing for numerical features
numerical_transformer = StandardScaler()

# Preprocessing for categorical features
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

In [8]:
final_model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)

In [9]:
final_model_pipeline.fit(X_train, y_train)

print("Final model pipeline trained successfully.")

Final model pipeline trained successfully.


In [10]:
y_pred = final_model_pipeline.predict(X_test)

print("Sample predictions:")
print(y_pred[:10])

Sample predictions:
['Average' 'Average' 'Average' 'Average' 'Average' 'Average' 'Average'
 'Average' 'Average' 'Average']


In [11]:
# Save the complete trained pipeline
joblib.dump(
    final_model_pipeline,
    "../models/random_forest_pipeline.joblib"
)

print("Model saved successfully.")

Model saved successfully.


In [12]:
# Load the saved model
loaded_model = joblib.load(
    "../models/random_forest_pipeline.joblib"
)

print("Saved model loaded successfully.")

Saved model loaded successfully.


In [13]:
# Make predictions using the loaded model
loaded_predictions = loaded_model.predict(X_test)

print("Predictions from loaded model:")
print(loaded_predictions[:10])

Predictions from loaded model:
['Average' 'Average' 'Average' 'Average' 'Average' 'Average' 'Average'
 'Average' 'Average' 'Average']


In [14]:
print(
    "Predictions are identical:",
    (y_pred == loaded_predictions).all()
)

Predictions are identical: True
